# NRAT Scraper — сбор PDF по годам (Google Colab)

Собирает PDF c nrat.ukrintei.ua. Год задаётся в ячейке «Настройки».
Файлы сохраняются на Google Drive: `MyDrive/nrat_pdfs/<год>/<месяц>/<день>/`.

## Порядок
1. **Ячейки 1–4** (Установка → Настройки → Функции) — запусти по очереди. На ячейке 1 разреши доступ к Google Drive.
2. **⬇️ ДОБРАТЬ КОНКРЕТНЫЕ ДНИ** — впиши даты (например, где была ошибка) и запусти. Скачает только их.
3. **СОБРАТЬ ВЕСЬ ГОД** — большой сбор на часы. Запускать, только если нужен весь год с нуля.
4. **СКАЧАТЬ АРХИВ** — упакует год в ZIP.

⚠️ Паузы между запросами не уменьшать — иначе бан на 1–2 дня.

In [ ]:
# ==============================================
# ЯЧЕЙКА 1 — установка, импорты, подключение Google Drive
# ==============================================
!pip install requests beautifulsoup4 tqdm -q

import requests
from bs4 import BeautifulSoup
import os
import time
import re
import json
from datetime import datetime, timedelta
from urllib.parse import urlencode
from tqdm import tqdm

# --- Подключаем Google Drive, чтобы НИЧЕГО не терялось при обрыве сессии ---
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive подключён")

In [ ]:
# ==============================================
# ЯЧЕЙКА 2 — КОНФИГУРАЦИЯ
# ==============================================

# >>> ГЛАВНОЕ: какой год собираем СЕЙЧАС. Меняй вручную: 2016, потом 2015, 2014 ... 1991 <<<
YEAR = 2016

# Куда сохраняем (прямо на Google Drive, чтобы пережить обрыв сессии)
DOWNLOAD_FOLDER = "/content/drive/MyDrive/nrat_pdfs"

# Чекпоинт у каждого года свой -> чистое продолжение после обрыва
CHECKPOINT_FILE = os.path.join(DOWNLOAD_FOLDER, f"_progress_{YEAR}.json")

# Поиск
BASE_SEARCH_URL = "https://nrat.ukrintei.ua/searchdb/?"
BASE_PARAMS = {
    '_token': '6C0elymE1XyE8AOayLEsuYco3JewHUAF0DazKx9Y',  # обновляется автоматически в ячейке 3
    'typeSearch2': 'ok',
    'typeCategory[]': '0',
    'lcSource': '',
    'authorSearch': '',
    'specialnistSearch[]': '0',
    'temaSearch2': '',
    'textSearch': '',
    'registrationNumberSearch': '',
    'firm_id': '0',
    'sortOrder': 'registration_date',
    'sortDir': 'desc',
    'tab': 'big'
}

DAYS_PER_CHUNK = 1   # 1 день за запрос — чтобы укладываться в лимит сайта (~1000 рез.)

# --- ПАУЗЫ (НЕ УМЕНЬШАТЬ! иначе бан на 1-2 дня) ---
DELAY_BETWEEN_PAGES = 3   # сек между страницами выдачи
DELAY_BETWEEN_FILES = 1   # сек между скачиванием PDF
DELAY_BETWEEN_DAYS  = 5   # сек между днями

os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)

def day_folder_for(year, date_from):
    """Структура как в существующих архивах: <год>/<год-месяц>/<дата>/"""
    return os.path.join(DOWNLOAD_FOLDER, str(year), date_from[:7], date_from)

# Сессия с заголовками браузера
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9,uk;q=0.8',
})

print(f"Собираем год: {YEAR}")
print("Папка на Drive:", os.path.join(DOWNLOAD_FOLDER, str(YEAR)))

In [ ]:
# ==============================================
# ЯЧЕЙКА 3 — ФУНКЦИИ
# ==============================================

# ---------- ЧЕКПОИНТ (продолжение после обрыва) ----------
def load_checkpoint():
    """Множество уже полностью обработанных дней (строки 'YYYY-MM-DD')."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
                return set(json.load(f).get('completed_dates', []))
        except Exception:
            return set()
    return set()

def save_checkpoint(completed_set):
    try:
        with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
            json.dump({'completed_dates': sorted(completed_set)}, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print(f"  ⚠️ Не удалось записать чекпоинт: {e}")

def mark_date_done(completed_set, date_str):
    """Помечаем день как готовый и сразу пишем чекпоинт на Drive."""
    completed_set.add(date_str)
    save_checkpoint(completed_set)

# ---------- АВТО-ОБНОВЛЕНИЕ _token ----------
def refresh_token():
    """Берём свежий _token со страницы поиска (на случай долгого прогона)."""
    try:
        r = session.get("https://nrat.ukrintei.ua/searchdb/", timeout=30)
        soup = BeautifulSoup(r.content, 'html.parser')
        tok = soup.find('input', attrs={'name': '_token'})
        if tok and tok.get('value'):
            BASE_PARAMS['_token'] = tok['value']
            print("  🔑 _token обновлён")
            return True
    except Exception as e:
        print(f"  ⚠️ Не удалось обновить _token ({e}), используем прежний")
    return False

# ---------- ДИАПАЗОНЫ ДАТ ----------
def generate_date_ranges(start_date_str, end_date_str, days_per_chunk):
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    ranges = []
    cur = start_date
    while cur <= end_date:
        cur_end = min(cur + timedelta(days=days_per_chunk - 1), end_date)
        ranges.append((cur.strftime("%Y-%m-%d"), cur_end.strftime("%Y-%m-%d")))
        cur = cur_end + timedelta(days=1)
    return ranges

def build_search_url(date_from, date_to, page=1):
    params = BASE_PARAMS.copy()
    params['dateFromSearch'] = date_from
    params['dateToSearch'] = date_to
    params['pa'] = str(page)   # ВАЖНО: на сайте параметр страницы называется 'pa', НЕ 'page'!
    return BASE_SEARCH_URL + urlencode(params, doseq=True)

# ---------- ЧИСЛО НАЙДЕННЫХ ДОКУМЕНТОВ ----------
def extract_total_results(soup):
    """Сколько документов нашлось ('Знайдено документів: N')."""
    try:
        page_info = soup.find('div', class_='page_info')
        if page_info:
            m = re.search(r'Знайдено документів:\s*(\d+)', page_info.get_text(strip=True))
            if m:
                return int(m.group(1))
        m = re.search(r'Знайдено документів:\s*(\d+)', soup.get_text(" ", strip=True))
        if m:
            return int(m.group(1))
    except Exception:
        pass
    return 0

# ---------- ПАРСИНГ СТРАНИЦЫ ВЫДАЧИ (с повторами при таймауте) ----------
def get_search_results_page(url, retry_count=3):
    """Возвращает (results, soup).
       results == None  -> страница НЕ загрузилась (ошибка сети) после всех попыток.
       results == []    -> страница загрузилась, но документов нет."""
    for attempt in range(retry_count):
        try:
            response = session.get(url, timeout=60)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')
            results = []
            for card in soup.find_all('div', class_='my-card-body'):
                link_tag = card.find('a', target='_blank')
                if link_tag and link_tag.get('href'):
                    reg_number = link_tag.get_text(strip=True)
                    detail_url = link_tag['href']
                    card_text = card.get_text(separator=' ', strip=True)
                    parts = card_text.split('Керівник:')
                    title = parts[0].strip() if len(parts) > 1 else card_text[:100].strip()
                    doc_id = detail_url.rstrip('/').split('/')[-1]
                    pdf_url = f"https://dir.ukrintei.ua/view/ok/{doc_id}"
                    results.append({
                        'registration': reg_number, 'detail_url': detail_url,
                        'pdf_url': pdf_url, 'title': title, 'doc_id': doc_id
                    })
            return results, soup
        except Exception as e:
            print(f"    ⚠️ загрузка выдачи, попытка {attempt + 1}/{retry_count}: {e}")
            if attempt < retry_count - 1:
                time.sleep(5)
    return None, None   # сеть так и не ответила

# ---------- СКАЧИВАНИЕ PDF ----------
def download_pdf(pdf_url, registration, title, doc_id, folder, retry_count=2):
    safe_reg = re.sub(r'[^\w\-_.]', '_', registration)
    filename = re.sub(r'_+', '_', f"{safe_reg}_{doc_id}.pdf")
    filepath = os.path.join(folder, filename)

    if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
        return 'skip', filepath

    for attempt in range(retry_count):
        try:
            response = session.get(pdf_url, stream=True, timeout=60)
            if response.status_code == 404:
                return 'notpdf', None   # у записи просто нет файла — не повторяем
            response.raise_for_status()
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            if os.path.getsize(filepath) > 0:
                with open(filepath, 'rb') as f:
                    if f.read(4).startswith(b'%PDF'):
                        return 'ok', filepath
                os.remove(filepath)
                return 'notpdf', None
            else:
                os.remove(filepath)
                return 'empty', None
        except Exception as e:
            print(f"    ❌ Ошибка скачивания: {e}")
            if attempt < retry_count - 1:
                time.sleep(3)
    return 'fail', None

print("✅ Функции загружены")

In [ ]:
# ==============================================
# ЯЧЕЙКА 4 — ОСНОВНАЯ ЛОГИКА
# ==============================================

def scrape_day(date_from, date_to, day_folder):
    """Скачать ВСЕ PDF за один день (все страницы). Возвращает (stats, day_ok).
       day_ok == False, если страница выдачи не загрузилась — день НЕ считать готовым."""
    os.makedirs(day_folder, exist_ok=True)
    page = 1
    page_size = None
    total_pages = None
    seen_ids = set()
    stats = {'ok': 0, 'skip': 0, 'fail': 0, 'notpdf': 0}
    day_ok = True

    while True:
        url = build_search_url(date_from, date_to, page)
        results, soup = get_search_results_page(url)

        if results is None:           # сеть не ответила после всех попыток
            print("    ⚠️ страница выдачи не загрузилась — день будет повторён позже")
            day_ok = False
            break

        if page == 1:
            total_results = extract_total_results(soup)
            page_size = len(results) if results else 10
            total_pages = ((total_results + 9) // 10) if total_results else None
            if total_results:
                print(f"    найдено {total_results} рез., {total_pages} стр.")
            if total_results >= 1000:
                print("    ⚠️ ВНИМАНИЕ: 1000+ результатов за день — возможно, не все попадут в выдачу")

        if not results:               # страница загрузилась, но документов нет
            if page == 1:
                print("    нет результатов")
            break

        new_results = [r for r in results if r['doc_id'] not in seen_ids]
        if page > 1 and not new_results:
            break
        for r in new_results:
            seen_ids.add(r['doc_id'])

        for i, r in enumerate(new_results, 1):
            status, _ = download_pdf(r['pdf_url'], r['registration'], r['title'], r['doc_id'], day_folder)
            stats[status if status in stats else 'fail'] += 1
            if i < len(new_results):
                time.sleep(DELAY_BETWEEN_FILES)

        if total_pages is not None:
            go_next = page < total_pages
        else:
            go_next = bool(page_size and len(results) >= page_size)
        if not go_next or page >= 85:
            break

        page += 1
        time.sleep(DELAY_BETWEEN_PAGES)

    return stats, day_ok


def run_year(year):
    """Собрать ОДИН год целиком (с продолжением после обрыва)."""
    completed = load_checkpoint()
    print(f"📌 По году {year} уже завершено дней: {len(completed)}")
    refresh_token()

    date_ranges = generate_date_ranges(f"{year}-01-01", f"{year}-12-31", DAYS_PER_CHUNK)
    grand = {'ok': 0, 'skip': 0, 'fail': 0, 'notpdf': 0}
    incomplete = []

    print("\n" + "=" * 70)
    print(f"ГОД {year} — {len(date_ranges)} дней")
    print("=" * 70)

    for idx, (date_from, date_to) in enumerate(date_ranges, 1):
        if date_from in completed:
            continue

        print(f"\n📅 {date_from}  ({idx}/{len(date_ranges)})")
        stats, day_ok = scrape_day(date_from, date_to, day_folder_for(year, date_from))
        for k in grand:
            grand[k] += stats[k]
        print(f"    итог дня: ✅{stats['ok']} ⏭{stats['skip']} 📄✗{stats['notpdf']} ❌{stats['fail']}"
              f"   |   ВСЕГО за год ✅{grand['ok']}")

        if day_ok:
            mark_date_done(completed, date_from)   # помечаем готовым ТОЛЬКО при успехе
        else:
            incomplete.append(date_from)
        time.sleep(DELAY_BETWEEN_DAYS)

    print("\n" + "=" * 70)
    print(f"ГОД {year} ГОТОВ!")
    print(f"Скачано новых PDF: {grand['ok']} | пропущено (уже было): {grand['skip']} | "
          f"без файла: {grand['notpdf']} | ошибок: {grand['fail']}")
    if incomplete:
        print(f"⚠️ Дней с ошибкой сети (НЕ завершены, запусти ячейку 6 ещё раз — доберутся): {len(incomplete)}")
        print("   ", incomplete)
    print("=" * 70)
    return grand

print("✅ Готово. Сначала тест (ячейка 5), потом полный год (ячейка 6).")

In [ ]:
# ==============================================
# ⬇️ ДОБРАТЬ КОНКРЕТНЫЕ ДНИ (например, где была ошибка Read timed out)
# Впиши даты ниже и запусти ТОЛЬКО эту ячейку. Качает РОВНО эти дни, НЕ весь год.
# Уже скачанные файлы пропускаются.
# ==============================================
DAYS_TO_FIX = [
    "2016-06-15",
    "2016-12-08",
    # добавь другие даты с ошибками, по одной в кавычках с запятой
]

refresh_token()
for d in DAYS_TO_FIX:
    print(f"\n📅 {d}")
    stats, ok = scrape_day(d, d, day_folder_for(YEAR, d))
    note = "" if ok else "  ⚠️ страница не догрузилась — запусти ещё раз"
    print(f"   ✅{stats['ok']} новых | ⏭{stats['skip']} уже было | 📄✗{stats['notpdf']} без файла | ❌{stats['fail']} ошибок{note}")
print("\n✅ Готово — обработаны ТОЛЬКО указанные дни.")

In [ ]:
# ==============================================
# СКАЧАТЬ АРХИВ ГОДА (ZIP) — запускать после сбора года
# Файлы и так лежат на Google Drive (nrat_pdfs/<год>/) — ZIP это просто
# удобный единый архив для скачивания на компьютер.
# ==============================================
import zipfile
from google.colab import files as colab_files

year_dir = os.path.join(DOWNLOAD_FOLDER, str(YEAR))
zip_path = f"/content/nrat_{YEAR}_pdfs_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"

count = 0
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, filenames in os.walk(year_dir):
        for fn in filenames:
            if fn.endswith('.pdf'):
                full = os.path.join(root, fn)
                arc = os.path.relpath(full, year_dir)   # внутри архива: <год-месяц>/<дата>/файл.pdf
                zf.write(full, arc)
                count += 1

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"✅ ZIP за {YEAR}: {count} PDF, {size_mb:.1f} MB")
print(f"   {zip_path}")
colab_files.download(zip_path)   # начнётся скачивание в браузере